In [ ]:
!pip install mantis-tsfm -q

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from google.colab import drive
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, classification_report
from mantis.architecture import MantisV2
from mantis.trainer import MantisTrainer

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/diss-datasets"

device = "cuda" if torch.cuda.is_available() else "cpu"
print('Device:', device)

network = MantisV2(device=device)
network = network.from_pretrained("paris-noah/MantisV2")
print('MantisV2 loaded')

def resize(X):
    return F.interpolate(
        torch.tensor(X, dtype=torch.float),
        size=512, mode="linear", align_corners=False
    ).numpy()

In [ ]:
data = np.load(f'{SAVE_DIR}/CHBMIT_crosspatient_v2.npy', allow_pickle=True).item()
X_tr = data['train']['X']
y_tr = np.array([int(x) for x in data['train']['y']])
X_te = data['test']['X']
y_te = np.array([int(x) for x in data['test']['y']])

X_tr_r = resize(X_tr)
X_te_r = resize(X_te)

model = MantisTrainer(device=device, network=network)
Z_tr = model.transform(X_tr_r)
Z_te = model.transform(X_te_r)

clf = RidgeClassifier()
clf.fit(Z_tr, y_tr)
y_pred = clf.predict(Z_te)

acc = accuracy_score(y_te, y_pred)
print(f'MantisV2 Accuracy on CHBMIT v2: {acc:.4f}')
print(classification_report(y_te, y_pred))

np.save(f'{SAVE_DIR}/MantisV2_CHBMIT_v2_train_emb.npy', Z_tr)
np.save(f'{SAVE_DIR}/MantisV2_CHBMIT_v2_test_emb.npy', Z_te)

In [ ]:
import os
SAVE_DIR = "/content/drive/MyDrive/diss-datasets"
for f in ['MantisV2_CHBMIT_v2_train_emb.npy', 'MantisV2_CHBMIT_v2_test_emb.npy',
          'Chronos2_CHBMIT_v2_train_emb.npy', 'Chronos2_CHBMIT_v2_test_emb.npy']:
    p = f'{SAVE_DIR}/{f}'
    print(f, '->', 'OK' if os.path.exists(p) else 'MISSING',
          f'({os.path.getsize(p)/1e6:.1f} MB)' if os.path.exists(p) else '')